In [1]:
%pip install kagglehub
import kagglehub
import os
import pandas as pd
import numpy as np
import sqlite3

Note: you may need to restart the kernel to use updated packages.


# Importing Dataset

In [2]:
# Download latest version
path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset") +'/'

print("Path to dataset files:", path)

df = pd.read_csv(path + "/dataset.csv")

Path to dataset files: C:\Users\avajt\.cache\kagglehub\datasets\maharshipandya\-spotify-tracks-dataset\versions\1/


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Unnamed: 0        114000 non-null  int64  
 1   track_id          114000 non-null  object 
 2   artists           113999 non-null  object 
 3   album_name        113999 non-null  object 
 4   track_name        113999 non-null  object 
 5   popularity        114000 non-null  int64  
 6   duration_ms       114000 non-null  int64  
 7   explicit          114000 non-null  bool   
 8   danceability      114000 non-null  float64
 9   energy            114000 non-null  float64
 10  key               114000 non-null  int64  
 11  loudness          114000 non-null  float64
 12  mode              114000 non-null  int64  
 13  speechiness       114000 non-null  float64
 14  acousticness      114000 non-null  float64
 15  instrumentalness  114000 non-null  float64
 16  liveness          11

In [4]:
# Drop unnecessary column
df = df.drop(columns=["Unnamed: 0"])

# Creating SQL Database

In [5]:
# Connect to SQLite database
conn = sqlite3.connect("spotify_dataset.db")
cursor = conn.cursor()

In [6]:
# Save dataframe to SQLite
df.to_sql("spotify_tracks", conn, if_exists = "replace", index = False)

114000

In [7]:
# List all tables in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

# Fetch and print table names
tables = cursor.fetchall()
print("Tables in database:", tables)

Tables in database: [('spotify_tracks',)]


In [8]:
with sqlite3.connect("spotify_dataset.db") as conn:
    df_tracks = pd.read_sql_query("SELECT * FROM spotify_tracks;", conn)
df_tracks.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,0,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,0,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,0,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,0,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,0,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


# Exploritatory Data Analysis

In [27]:
def get_numerical_df(df):
  """
  Returns a dataframe with only the numerical data type columns in a Pandas DataFrame.
  Parameters:
    df: The Pandas DataFrame.
  Returns:
    A Pandas datafram with the numerical columns from df.
  """
  numerical_cols = df.select_dtypes(include=['number']).columns.tolist()
  num_df = df[numerical_cols]
  return num_df

df_num_cols = get_numerical_df(df_tracks)
df_num_cols.head()

,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
0,73,230666,0,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4
1,55,149610,0,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4
2,57,210826,0,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4
3,71,201933,0,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3
4,82,198853,0,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4


In [28]:
corr_matrix = df_num_cols.corr()

In [29]:
from plotly import express as px
fig = px.imshow(corr_matrix,
                x=corr_matrix.columns,
                y=corr_matrix.index,
                title="Correlation Matrix Heatmap")

fig.update_layout(
    xaxis_title="",
    yaxis_title="",
    width=800,
    height=800
)

fig.show()